In [ ]:
!pip install ultralytics easyocr opencv-python

from google.colab import files
uploaded = files.upload()

video_path = list(uploaded.keys())[0]
print("Using video:", video_path)

In [ ]:
from ultralytics import YOLO
import cv2
import easyocr
import re
from collections import Counter
from google.colab.patches import cv2_imshow
from IPython.display import Video

model = YOLO("yolov8n.pt")

reader = easyocr.Reader(['en'])

cap = cv2.VideoCapture(video_path)

width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = int(cap.get(cv2.CAP_PROP_FPS))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('output.mp4', fourcc, fps, (width, height))

detected_numbers = []
frame_count = 0

In [ ]:
def clean_plate(text):
    text = re.sub(r'[^A-Z0-9]', '', text.upper())

    if len(text) < 6:
        return None

    return text

In [ ]:
while True:
    ret, frame = cap.read()

    if not ret:
        print("✅ Video finished")
        break

    frame_count += 1

    if frame_count % 5 != 0:
        out.write(frame)
        continue

    results = model(frame)

    for box in results[0].boxes:
        cls = int(box.cls[0])
        conf = float(box.conf[0])

        if cls not in [2, 3, 5, 7] or conf < 0.5:
            continue

        x1, y1, x2, y2 = map(int, box.xyxy[0])
        vehicle_crop = frame[y1:y2, x1:x2]

        if vehicle_crop.size == 0:
            continue

        h, w, _ = vehicle_crop.shape

        plate_crop = vehicle_crop[int(h*0.5):h, int(w*0.15):int(w*0.85)]

        gray = cv2.cvtColor(plate_crop, cv2.COLOR_BGR2GRAY)
        gray = cv2.bilateralFilter(gray, 11, 17, 17)
        gray = cv2.equalizeHist(gray)

        try:
            ocr_result = reader.readtext(gray)

            for detection in ocr_result:
                raw_text = detection[1]
                confidence = detection[2]

                if confidence < 0.4:
                    continue

                plate_text = clean_plate(raw_text)

                if plate_text:
                    detected_numbers.append(plate_text)

                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)
                    cv2.putText(frame, plate_text, (x1, y1-10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)

        except:
            continue

    out.write(frame)

    if frame_count % 30 == 0:
        print(f"Frame {frame_count}")
        cv2_imshow(frame)

In [ ]:
cap.release()
out.release()

Video("output.mp4")

In [ ]:
print("\n All Detected:")
print(detected_numbers)

counter = Counter(detected_numbers)

final_plates = [plate for plate, count in counter.items() if count >= 2]

print("\n Final Stable Plates:")
print(final_plates)